In [ ]:

import time
import requests
import pandas as pd
from pathlib import Path

BASE_URL = "https://api.tickermint.cards"
OUTPUT_FILE = Path("riftbound_all_price_history.csv")
PROGRESS_FILE = Path("riftbound_processed_ids.txt")

session = requests.Session()
session.headers.update({
    "User-Agent": "RiftboundMarketResearch/1.0"
})

def get_json(url, params=None):
    """Request JSON with basic retry handling."""
    for attempt in range(5):
        response = session.get(url, params=params, timeout=45)

        if response.status_code == 429:
            wait = int(response.headers.get("Retry-After", 10))
            print(f"Rate limited; waiting {wait} seconds...")
            time.sleep(wait)
            continue

        response.raise_for_status()
        return response.json()

    raise RuntimeError(f"Too many retries: {url}")


def get_records(data):
    """Normalize common API response shapes into a list of dictionaries."""
    if isinstance(data, list):
        return data

    if isinstance(data, dict):
        for key in ("results", "data", "sets", "items"):
            if isinstance(data.get(key), list):
                return data[key]

    raise ValueError(
        f"Unexpected API response structure: {str(data)[:500]}"
    )


# 1. Discover all Riftbound sets
sets_data = get_json(
    f"{BASE_URL}/sets",
    params={"game": "riftbound"}
)
sets_list = get_records(sets_data)

print(f"Found {len(sets_list)} sets")

# 2. Download each set's current-price CSV to discover product IDs
products = {}

for set_info in sets_list:
    group_id = (
        set_info.get("group_id")
        or set_info.get("id")
        or set_info.get("set_id")
    )

    if group_id is None:
        print("Skipping set with unknown ID:", set_info)
        continue

    url = f"{BASE_URL}/sets/{group_id}.csv"

    try:
        response = session.get(url, timeout=45)
        response.raise_for_status()

        from io import StringIO
        set_df = pd.read_csv(StringIO(response.text))

        if "product_id" not in set_df.columns:
            print(f"Missing product_id in set {group_id}")
            continue

        for _, row in set_df.iterrows():
            pid = int(row["product_id"])
            products[pid] = row.to_dict()

        print(f"Set {group_id}: {len(set_df)} products")

    except Exception as exc:
        print(f"Failed set {group_id}: {exc}")

    time.sleep(0.7)

print(f"\nUnique products discovered: {len(products)}")

# 3. Load progress so interrupted runs can resume
if PROGRESS_FILE.exists():
    processed = set(
        PROGRESS_FILE.read_text().splitlines()
    )
else:
    processed = set()

print(f"Already processed: {len(processed)}")

# 4. Retrieve daily price history for every product
total = len(products)

for index, (product_id, metadata) in enumerate(products.items(), start=1):
    pid = str(product_id)

    if pid in processed:
        continue

    try:
        history_data = get_json(
            f"{BASE_URL}/products/{product_id}/prices"
        )

        records = []

        # API returns a separate list for each printing/variant
        if isinstance(history_data, dict):
            for printing, observations in history_data.items():
                if not isinstance(observations, list):
                    continue

                for observation in observations:
                    records.append({
                        **metadata,
                        "product_id": product_id,
                        "printing": printing,
                        **observation,
                        "tickerMint_url":
                            f"https://tickermint.cards/card/{product_id}"
                    })

        if records:
            history_df = pd.DataFrame(records)

            # Append immediately so progress isn't lost
            history_df.to_csv(
                OUTPUT_FILE,
                mode="a",
                header=not OUTPUT_FILE.exists(),
                index=False
            )

        # Mark complete even if no observations were returned
        with PROGRESS_FILE.open("a", encoding="utf-8") as f:
            f.write(pid + "\n")

        processed.add(pid)

        if index % 25 == 0:
            print(
                f"Processed {index}/{total} products; "
                f"{len(processed)} completed"
            )

    except Exception as exc:
        print(f"Failed product {product_id}: {exc}")

    # Stay below the documented free API limit
    time.sleep(0.65)

print("\nCollection finished.")
print(f"History CSV: {OUTPUT_FILE.resolve()}")
print(f"Processed IDs: {PROGRESS_FILE.resolve()}")

# 5. Inspect the resulting dataset
if OUTPUT_FILE.exists():
    df = pd.read_csv(OUTPUT_FILE)

    if "snapshot_date" in df.columns:
        df["snapshot_date"] = pd.to_datetime(
            df["snapshot_date"], errors="coerce"
        )

    print(df.head())
    print(f"\nRows: {len(df):,}")
    print(f"Products: {df['product_id'].nunique():,}")
    print(df.columns.tolist())


Found 14 sets
Set 24832: 7 products
Set 24901: 1 products
Set 24819: 201 products
Set 24861: 17 products
Set 24698: 258 products
Set 24560: 322 products
Set 24519: 323 products
Set 24552: 4 products
Set 24528: 240 products
Set 24343: 12 products
Set 24439: 30 products
Set 24344: 365 products
Set 24502: 8 products
Set 24797: 4 products

Unique products discovered: 1792
Already processed: 0
Processed 25/1792 products; 25 completed
Processed 50/1792 products; 50 completed
Processed 75/1792 products; 75 completed
Processed 100/1792 products; 100 completed
Processed 125/1792 products; 125 completed
Processed 150/1792 products; 150 completed
Processed 175/1792 products; 175 completed
Processed 200/1792 products; 200 completed
Processed 225/1792 products; 225 completed
Processed 250/1792 products; 250 completed
Processed 275/1792 products; 275 completed
Processed 300/1792 products; 300 completed
Processed 325/1792 products; 325 completed
Processed 350/1792 products; 350 completed
Processed 37

In [ ]:
df

,productId,name,cleanName,imageUrl,categoryId,groupId,url,modifiedOn,imageCount,presaleInfo,extendedData,lowPrice,midPrice,highPrice,marketPrice,directLowPrice,subTypeName
0,655495,Riftbound: League of Legends Arcane Box Set,Riftbound League of Legends Arcane Box Set,https://tcgplayer-cdn.tcgplayer.com/product/65...,89,24343,https://www.tcgplayer.com/product/655495/riftb...,2026-02-11T05:10:10.187,1,"{'isPresale': False, 'releasedOn': '2026-02-10...",[],486.00,794.99,3899.00,528.12,None,Normal
1,659765,"Ahri, Alluring (Launch Exclusive)",Ahri Alluring Launch Exclusive,https://tcgplayer-cdn.tcgplayer.com/product/65...,89,24343,https://www.tcgplayer.com/product/659765/riftb...,2026-08-12T20:25:32.503,1,"{'isPresale': False, 'releasedOn': None, 'note...","[{'name': 'Rarity', 'displayName': 'Rarity', '...",124.99,200.00,999.99,139.83,None,Foil
2,659766,"Teemo, Scout",Teemo Scout,https://tcgplayer-cdn.tcgplayer.com/product/65...,89,24343,https://www.tcgplayer.com/product/659766/riftb...,2026-08-12T20:25:32.613,1,"{'isPresale': False, 'releasedOn': None, 'note...","[{'name': 'Rarity', 'displayName': 'Rarity', '...",6.04,10.78,89.99,9.10,None,Normal
3,663004,Buff // Buff (Fist Bump Promo),Buff Buff Fist Bump Promo,https://tcgplayer-cdn.tcgplayer.com/product/66...,89,24343,https://www.tcgplayer.com/product/663004/riftb...,2026-01-08T18:26:00.66,0,"{'isPresale': False, 'releasedOn': None, 'note...","[{'name': 'Rarity', 'displayName': 'Rarity', '...",139.78,172.15,350.00,148.48,None,Foil
4,678049,"Warwick, Hunter",Warwick Hunter,https://tcgplayer-cdn.tcgplayer.com/product/67...,89,24343,https://www.tcgplayer.com/product/678049/riftb...,2026-08-12T20:25:32.763,1,"{'isPresale': False, 'releasedOn': None, 'note...","[{'name': 'Rarity', 'displayName': 'Rarity', '...",56.91,62.94,599.99,68.69,None,Foil
5,678050,"Caitlyn, Patrolling",Caitlyn Patrolling,https://tcgplayer-cdn.tcgplayer.com/product/67...,89,24343,https://www.tcgplayer.com/product/678050/riftb...,2026-08-12T20:25:32.89,1,"{'isPresale': False, 'releasedOn': None, 'note...","[{'name': 'Rarity', 'displayName': 'Rarity', '...",164.90,224.50,1099.99,103.62,None,Foil
6,678051,"Heimerdinger, Inventor",Heimerdinger Inventor,https://tcgplayer-cdn.tcgplayer.com/product/67...,89,24343,https://www.tcgplayer.com/product/678051/riftb...,2026-08-12T20:25:33.083,1,"{'isPresale': False, 'releasedOn': None, 'note...","[{'name': 'Rarity', 'displayName': 'Rarity', '...",69.99,79.95,599.99,80.19,None,Foil
7,678052,"Jinx, Rebel",Jinx Rebel,https://tcgplayer-cdn.tcgplayer.com/product/67...,89,24343,https://www.tcgplayer.com/product/678052/riftb...,2026-08-12T20:25:33.143,1,"{'isPresale': False, 'releasedOn': None, 'note...","[{'name': 'Rarity', 'displayName': 'Rarity', '...",243.99,249.44,899.99,251.39,None,Foil
8,678053,"Vi, Destructive",Vi Destructive,https://tcgplayer-cdn.tcgplayer.com/product/67...,89,24343,https://www.tcgplayer.com/product/678053/riftb...,2026-08-12T20:25:34.043,1,"{'isPresale': False, 'releasedOn': None, 'note...","[{'name': 'Rarity', 'displayName': 'Rarity', '...",98.90,102.40,899.99,93.81,None,Foil
9,678054,"Viktor, Leader",Viktor Leader,https://tcgplayer-cdn.tcgplayer.com/product/67...,89,24343,https://www.tcgplayer.com/product/678054/riftb...,2026-08-12T20:25:34.123,1,"{'isPresale': False, 'releasedOn': None, 'note...","[{'name': 'Rarity', 'displayName': 'Rarity', '...",108.99,141.98,899.99,110.08,None,Foil
